In [ ]:
pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 7.4 MB/s eta 0:00:00


In [ ]:
pip install pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=dc85dacc1ce7b68d34a495bfc152989d175e951a531bf317914bd924a679f0b4
  Stored in directory: /root/.cache/pip/wheels/b1/7a/33/9fdd892f784ed4afda62b685ae3703adf4c91aa0f524c28f03
Successfully built pylatexenc


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister
from qiskit.circuit.library import QFT, RZZGate, RZGate, UnitaryGate, DiagonalGate
from qiskit.circuit.library.generalized_gates import Diagonal
from qiskit.quantum_info import random_unitary
from qiskit.circuit import Gate
from qiskit.quantum_info import Statevector, Operator

In this lab, we will implement the ISHE flow simulation of the 1 dimensional ISF flow. Note that the only differences between different dimensions are the initial encoding step. The quantum algorithm has 3 subcircuits: Prediction, Normalization, and Gauge transformation.

In [ ]:
def discretize_domain_with_binary_vectors(d, n):
    num_points = 2**n
    delta_x = (2 * d) / num_points
    j_values = np.arange(num_points)
    x_j = -d + (j_values + 0.5) * delta_x
    binary_vectors = np.zeros((num_points, n), dtype=int)
    for i in range(n):
        binary_vectors[:, i] = (j_values // (2**i)) % 2
    return x_j, binary_vectors

def construct_quaternionic_psi_statevector(d, n_spatial_qubits, psi_coefficients_map, time_t):
    """
    Constructs the state vector |psi> as described in equation (36) using Qiskit.
    It returns the augmented statevector, matching common practice.
    """
    x_points, _ = discretize_domain_with_binary_vectors(d, n_spatial_qubits)

    n_s_and_j_qubits = n_spatial_qubits + 1

    state_amplitudes_original = np.zeros(2**n_s_and_j_qubits, dtype=complex)

    for s_val in range(2):
        for j_idx in range(2**n_spatial_qubits):
            x_j_current = x_points[j_idx]
            psi_coeff = psi_coefficients_map(s_val, x_j_current, time_t)
            combined_index = (s_val << n_spatial_qubits) | j_idx
            state_amplitudes_original[combined_index] = psi_coeff

    initial_psi_statevector = Statevector(state_amplitudes_original)

    original_amplitudes = initial_psi_statevector.data
    original_num_qubits = initial_psi_statevector.num_qubits
    new_num_qubits = original_num_qubits + 1
    new_dimension = 2**new_num_qubits
    augmented_amplitudes = np.zeros(new_dimension, dtype=complex)
    augmented_amplitudes[::2] = original_amplitudes
    augmented_statevector = Statevector(augmented_amplitudes)
    return augmented_statevector

In [ ]:
def my_psi_coefficients_func(s_val, binary_j_list, current_time):
    # Example: A simple function where coefficients depend on s_val and spatial index
    # You would replace this with your actual wave function calculation!
    spatial_index = int("".join(map(str, binary_j_list)), 2) # Convert binary list to integer

    if s_val == 0: # Corresponds to first component of quaternion
        return (spatial_index + 1) * np.exp(1j * current_time) + 0.5j
    else: # Corresponds to second component of quaternion
        return (spatial_index * 0.5 - 1) * np.exp(-1j * current_time * 0.5) - 0.2j



**1. Step 1: Prediction**

In [ ]:
class PK2dtGate(Gate):
    def __init__(self, phi_angle: float, theta_angle: float, n_spatial_qubits: int):
        self.n_spatial_qubits = n_spatial_qubits
        self.phi_angle = phi_angle
        self.theta_angle = theta_angle
        super().__init__('P(k^2dt)', n_spatial_qubits + 2, [], f"P(k^2dt)_n{n_spatial_qubits}")
    def _define(self):
        qr_s = QuantumRegister(1, 's')
        qr_j = QuantumRegister(self.n_spatial_qubits, 'j')
        qr_anc = AncillaRegister(1, 'anc')
        sub_qc = QuantumCircuit(qr_s, qr_j, qr_anc, name='pk2dt_internal')
        for l in range(self.n_spatial_qubits):
            j_l_qubit = qr_j[l]
            alpha_l = self.theta_angle / (2**(2 * self.n_spatial_qubits - l - 4))
            sub_qc.rz(-2 * alpha_l, j_l_qubit)
        for i in range(self.n_spatial_qubits):
            for j_idx_inner in range(i):
                j_i_qubit = qr_j[i]
                j_j_qubit = qr_j[j_idx_inner]
                beta_ij = self.theta_angle / (2**(2 * self.n_spatial_qubits - i - j_idx_inner))
                rz_angle_on_ancilla = -2 * beta_ij
                sub_qc.cx(j_i_qubit, qr_anc[0])
                sub_qc.cx(j_j_qubit, qr_anc[0])
                sub_qc.rz(rz_angle_on_ancilla, qr_anc[0])
                sub_qc.cx(j_j_qubit, qr_anc[0])
                sub_qc.cx(j_i_qubit, qr_anc[0])
        self.definition = sub_qc

class PQxGate(Gate):
    def __init__(self, q_function, d: float, n_spatial_qubits: int, h_reduced: float):
        self.q_function = q_function
        self.d = d
        self.n_spatial_qubits = n_spatial_qubits
        self.h_reduced = h_reduced
        super().__init__('P(q(x))', n_spatial_qubits, [], f"P(q(x))_n{n_spatial_qubits}")
    def _define(self):
        qr_j = QuantumRegister(self.n_spatial_qubits, 'j')
        sub_qc = QuantumCircuit(qr_j, name='pqx_internal')
        x_points, _ = discretize_domain_with_binary_vectors(self.d, self.n_spatial_qubits)
        num_j_states = 2**self.n_spatial_qubits
        diagonal_phases = []
        for j_idx in range(num_j_states):
            x_j_current = x_points[j_idx]
            q_val = self.q_function(x_j_current)
            phase_angle = -q_val / self.h_reduced
            diagonal_phases.append(np.exp(1j * phase_angle))
        diagonal_gate = DiagonalGate(diagonal_phases)
        sub_qc.append(diagonal_gate, qr_j[:])
        self.definition = sub_qc

# --- Corrected P(V_F dt) Gate for Linear Potential ---
class PVFdtGate(Gate):
    """
    Custom gate for the P(V_F*delta_t) operator, where V_F is a LINEAR potential V_F(x) = C_linear * x.
    This gate applies phases exp(-i * delta_t / h_reduced * V_F(x_j)).
    It acts ONLY on the spatial qubits |j_0>...|j_{n-1}>.
    """
    def __init__(self, C_linear: float, delta_t: float, h_reduced: float, d: float, n_spatial_qubits: int):
        self.C_linear = C_linear
        self.delta_t = delta_t
        self.h_reduced = h_reduced
        self.d = d
        self.n_spatial_qubits = n_spatial_qubits

        # The gate acts on n_spatial_qubits, not n_spatial_qubits + 1
        super().__init__('P(V_Fdt)', n_spatial_qubits, [], f"P(V_Fdt)_linear_n{n_spatial_qubits}")

    def _define(self):
        qr_j = QuantumRegister(self.n_spatial_qubits, 'j')
        sub_qc = QuantumCircuit(qr_j, name='pvfdt_linear_internal')

        delta_x = (2 * self.d) / (2**self.n_spatial_qubits)

        # This decomposition works by applying RZ gates to the spatial qubits
        # to generate a phase proportional to x_j
        lambda_coeff = - (self.delta_t * self.C_linear / self.h_reduced) * delta_x

        for k in range(self.n_spatial_qubits):
            j_k_qubit = qr_j[k]
            rz_angle = 2 * (lambda_coeff * (2**k))
            sub_qc.rz(rz_angle, j_k_qubit)

        # The constant part of the potential contributes a global phase
        gamma_constant_part = - (self.delta_t * self.C_linear / self.h_reduced)
        global_phase_angle = gamma_constant_part * (-self.d + 0.5 * delta_x)

        sub_qc.global_phase = global_phase_angle

        self.definition = sub_qc



2. Step 2,3,4: Normalization and Gauge transformation

In [ ]:
def apply_un_normalization_to_statevector(
    current_statevector: Statevector,
    n_spatial_qubits: int
) -> Statevector:
    new_amplitudes = np.copy(current_statevector.data)
    total_num_qubits = current_statevector.num_qubits
    num_other_qubits = total_num_qubits - (1 + n_spatial_qubits)
    num_j_states = 2**n_spatial_qubits
    for other_val_decimal in range(2**num_other_qubits):
        for j_idx in range(num_j_states):
            index_s0 = (other_val_decimal << (n_spatial_qubits + 1)) | (j_idx << 1) | 0
            index_s1 = (other_val_decimal << (n_spatial_qubits + 1)) | (j_idx << 1) | 1
            amp_s0_current = new_amplitudes[index_s0]
            amp_s1_current = new_amplitudes[index_s1]
            norm_sq = np.abs(amp_s0_current)**2 + np.abs(amp_s1_current)**2
            if np.isclose(norm_sq, 0.0):
                continue
            norm_factor = np.sqrt(norm_sq)
            new_amplitudes[index_s0] /= norm_factor
            new_amplitudes[index_s1] /= norm_factor
    return Statevector(new_amplitudes)._data_setter(new_amplitudes / np.linalg.norm(new_amplitudes))


In [ ]:
def build_full_quantum_step(
    n_spatial_qubits: int,
    C_linear_vf: float, delta_t: float, h_reduced: float, d: float,
    phi_angle_pk2dt: float, theta_angle_pk2dt: float,
    q_function
) -> QuantumCircuit:
    """
    Constructs a full quantum circuit segment with corrected qubit assignments.
    """
    qr_s = QuantumRegister(1, 's')
    qr_j = QuantumRegister(n_spatial_qubits, 'j')
    qr_anc = AncillaRegister(1, 'anc')

    qc = QuantumCircuit(qr_s, qr_j, qr_anc)

    # 1. Corrected P(V_F dt) operator: Acts ONLY on the spatial qubits
    pvfdt_gate = PVFdtGate(C_linear_vf, delta_t, h_reduced, d, n_spatial_qubits)
    qc.append(pvfdt_gate, qr_j[:])

    # 2. QFT on |j_0>,...,|j_{n-1}>
    qc.append(QFT(n_spatial_qubits), qr_j)

    # 3. P(k^2 dt) operator
    pk2dt_gate = PK2dtGate(phi_angle_pk2dt, theta_angle_pk2dt, n_spatial_qubits)
    qc.append(pk2dt_gate, [qr_s[0]] + qr_j[:] + [qr_anc[0]])

    # 4. Inverse QFT on |j_0>,...,|j_{n-1}>
    qc.append(QFT(n_spatial_qubits).inverse(), qr_j)

    # 5. P(q(x_j)) operator
    pqx_gate = PQxGate(q_function, d, n_spatial_qubits, h_reduced)
    qc.append(pqx_gate, qr_j[:])

    return qc

In [ ]:
def build_full_quantum_step(
    n_spatial_qubits: int,
    C_linear_vf: float, delta_t: float, h_reduced: float, d: float,
    phi_angle_pk2dt: float, theta_angle_pk2dt: float,
    q_function # The function q(x) for P(q(x))
) -> QuantumCircuit:
    """
    Constructs a full quantum circuit segment combining previous steps and the new P(q(x)) gate.

    Qubit Mapping (LSB to MSB, consistent with previous code):
    q_0: 's' qubit
    q_1 to q_n_spatial_qubits: 'j_0' to 'j_{n_spatial_qubits-1}' qubits (j-register)
    q_{n_spatial_qubits + 1}: 'ancilla' qubit (ancilla register)
    """
    qr_s = QuantumRegister(1, 's')
    qr_j = QuantumRegister(n_spatial_qubits, 'j')
    qr_anc = AncillaRegister(1, 'anc')

    qc = QuantumCircuit(qr_s, qr_j, qr_anc)

    # 1. P(V_F dt) operator
    pvfdt_gate = PVFdtGate(C_linear_vf, delta_t, h_reduced, d, n_spatial_qubits)
    qc.append(pvfdt_gate, [qr_s[0]] + qr_j[:])

    # 2. QFT on |j_0>,...,|j_{n-1}>
    qc.append(QFT(n_spatial_qubits), qr_j)

    # 3. P(k^2 dt) operator
    pk2dt_gate = PK2dtGate(phi_angle_pk2dt, theta_angle_pk2dt, n_spatial_qubits)
    qc.append(pk2dt_gate, [qr_s[0]] + qr_j[:] + [qr_anc[0]])

    # 4. Inverse QFT on |j_0>,...,|j_{n-1}>
    qc.append(QFT(n_spatial_qubits).inverse(), qr_j)

    # 5. P(q(x_j)) operator (newly added)
    # This gate acts *only* on the spatial qubits (j register).
    pqx_gate = PQxGate(q_function, d, n_spatial_qubits, h_reduced)
    qc.append(pqx_gate, qr_j[:])

    return qc


In [ ]:
def extract_psi_components_from_statevector(
    statevector: np.ndarray,
    n_spatial_qubits: int
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extracts the psi_0 and psi_1 components of the wave function from the statevector,
    assuming the ancilla is in the |0> state.
    """
    num_j_states = 2**n_spatial_qubits
    psi_0_coeffs = np.zeros(num_j_states, dtype=complex)
    psi_1_coeffs = np.zeros(num_j_states, dtype=complex)

    for j_idx in range(num_j_states):
        # Index for |s=0, j_idx, ancilla=0>
        index_s0 = (0 << (n_spatial_qubits + 1)) | (j_idx << 1) | 0
        psi_0_coeffs[j_idx] = statevector[index_s0]

        # Index for |s=1, j_idx, ancilla=0>
        index_s1 = (0 << (n_spatial_qubits + 1)) | (j_idx << 1) | 1
        psi_1_coeffs[j_idx] = statevector[index_s1]

    return psi_0_coeffs, psi_1_coeffs


def calculate_quantities(
    psi_0: np.ndarray,
    psi_1: np.ndarray,
    x_points: np.ndarray,
    h_reduced: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calculates rho, u, and omega from the extracted psi components.
    """
    if len(psi_0) != len(psi_1):
        raise ValueError("Psi_0 and Psi_1 must have the same length.")

    delta_x = x_points[1] - x_points[0]

    psi_0_conj = np.conjugate(psi_0)
    psi_1_conj = np.conjugate(psi_1)

    rho = psi_0_conj * psi_0 + psi_1_conj * psi_1

    # Calculate gradient of psi_0 and psi_1 using central finite difference
    grad_psi_0 = np.zeros_like(psi_0, dtype=complex)
    grad_psi_1 = np.zeros_like(psi_1, dtype=complex)

    grad_psi_0[1:-1] = (psi_0[2:] - psi_0[0:-2]) / (2 * delta_x)
    grad_psi_1[1:-1] = (psi_1[2:] - psi_1[0:-2]) / (2 * delta_x)

    # Forward difference for the first point
    grad_psi_0[0] = (psi_0[1] - psi_0[0]) / delta_x
    grad_psi_1[0] = (psi_1[1] - psi_1[0]) / delta_x

    # Backward difference for the last point
    grad_psi_0[-1] = (psi_0[-1] - psi_0[-2]) / delta_x
    grad_psi_1[-1] = (psi_1[-1] - psi_1[-2]) / delta_x

    # Calculate the numerator of u
    numerator = 1j * (
        (np.conjugate(grad_psi_0) * psi_0) + (np.conjugate(grad_psi_1) * psi_1) -
        (psi_0_conj * grad_psi_0) - (psi_1_conj * grad_psi_1)
    )

    # Avoid division by zero where rho is near zero
    u = np.zeros_like(rho, dtype=complex)
    non_zero_rho = np.abs(rho) > 1e-10
    u[non_zero_rho] = (h_reduced / 2) * (numerator[non_zero_rho] / rho[non_zero_rho])

    # Curl of a 1D vector field is zero
    omega = np.zeros_like(u)

    return np.real(rho), np.real(u), np.real(omega)

In [ ]:
if __name__ == '__main__':
    n_spatial_qubits_example = 2
    d_example = 10.0
    h_reduced_example = 1.0

    x_points, _ = discretize_domain_with_binary_vectors(d_example, n_spatial_qubits_example)

    num_total_qubits = n_spatial_qubits_example + 2
    dummy_amplitudes = np.zeros(2**num_total_qubits, dtype=complex)

    for j_idx, x_val in enumerate(x_points):
        psi_0_val = np.exp(-(x_val / 5)**2)
        psi_1_val = np.sin(x_val / 5) * np.exp(1j * x_val)

        index_s0 = (0 << (n_spatial_qubits_example + 1)) | (j_idx << 1) | 0
        index_s1 = (0 << (n_spatial_qubits_example + 1)) | (j_idx << 1) | 1

        dummy_amplitudes[index_s0] = psi_0_val
        dummy_amplitudes[index_s1] = psi_1_val

    dummy_statevector = Statevector(dummy_amplitudes)

    print(f"Dummy Statevector Norm: {np.linalg.norm(dummy_statevector.data):.4f}")

    psi_0_extracted, psi_1_extracted = extract_psi_components_from_statevector(
        dummy_statevector.data,
        n_spatial_qubits_example
    )

    print("\nExtracted Psi_0 components:")
    print(psi_0_extracted)
    print("\nExtracted Psi_1 components:")
    print(psi_1_extracted)

    rho_calculated, u_calculated, omega_calculated = calculate_quantities(
        psi_0_extracted,
        psi_1_extracted,
        x_points,
        h_reduced_example
    )

    print("\nCalculated physical quantities:")
    print(f"rho (probability density): {rho_calculated}")
    print(f"u (velocity): {u_calculated}")
    print(f"omega (vorticity): {omega_calculated}")

Dummy Statevector Norm: 1.9196

Extracted Psi_0 components:
[0.10539922+0.j 0.77880078+0.j 0.77880078+0.j 0.10539922+0.j]

Extracted Psi_1 components:
[-0.34576699+0.93565027j  0.38408871+0.28692283j -0.38408871+0.28692283j
  0.34576699+0.93565027j]

Calculated physical quantities:
rho (probability density): [1.00610524 0.83637951 0.83637951 1.00610524]
u (velocity): [-0.09115968 -0.02847673 -0.02847673 -0.09115968]
omega (vorticity): [0. 0. 0. 0.]
